# 🔄 Basic Agent Workflows wit Azure OpenAI (Responses API) (.NET)

## 📋 Workflow Orchestration Tutorial

Dis notebook show how to build advanced **agent workflows** usin di Microsoft Agent Framework for .NET and Azure OpenAI (Responses API). You go learn how to create multi-step business processes wey AI agents dey work together to finish complex tasks through organized orchestration patterns.

## 🎯 Wetin You Go Learn

### 🏗️ **Workflow Architecture Basics**
- **Workflow Builder**: Design and organize complex multi-step AI processes
- **Agent Coordination**: Arrange plenty specialized agents inside workflows
- **Azure OpenAI (Responses API)**: Use Azure OpenAI Responses API for workflows
- **Visual Workflow Design**: Create and visualize workflow structures make e clear

### 🔄 **Process Orchestration Patterns**
- **Sequential Processing**: Join many agent tasks for logical order
- **State Management**: Keep context and data flow for all workflow stages
- **Error Handling**: Do strong error correction and make workflow strong
- **Performance Optimization**: Design efficient workflows for big business operations

### 🏢 **Enterprise Workflow Applications**
- **Business Process Automation**: Make complex company workflows automatic
- **Content Production Pipeline**: Editorial workflows with review and approval steps
- **Customer Service Automation**: Multi-step solution for customer questions
- **Data Processing Workflows**: ETL workflows wey get AI-powered transformation

## ⚙️ Wetin You Need & Setup

### 📦 **Wetin NuGet Packages You Need**

Dis workflow demo dey use some important .NET packages:

```xml
<!-- Core AI Framework -->
<PackageReference Include="Microsoft.Extensions.AI" Version="10.*" />

<!-- Azure OpenAI (Responses API) -->
<PackageReference Include="Azure.AI.OpenAI" Version="2.1.0" />
<PackageReference Include="Azure.Identity" Version="1.15.0" />

<!-- Agent Framework (Local Development) -->
<!-- Microsoft.Agents.AI.dll - Core agent abstractions -->
<!-- Microsoft.Agents.AI.OpenAI.dll - Azure OpenAI (Responses API) integration -->

<!-- Configuration and Environment -->
<PackageReference Include="DotNetEnv" Version="3.1.1" />
```

### 🔑 **Azure OpenAI Setup**

**Environment Setup (.env file):**
```env
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT=gpt-5-mini
```

**Azure OpenAI Access:**
1. Create Azure OpenAI resource for Azure portal
2. Deploy one model (example, `gpt-5-mini`) and remember the deployment name
3. Sign in with `az login` and set environment variables as e dey above

### 🏗️ **Workflow Architecture Overview**

```mermaid
graph TD
    A[Workflow Buidah] --> B[Agent Registree]
    B --> C[Workflow Ejikushon Engine]
    C --> D[Agent 1: Content Genereita]
    C --> E[Agent 2: Content Reviewa] 
    D --> F[Workflow Result dem]
    E --> F
    G[Azure OpenAI (Responses API)] --> D
    G --> E
```

**Main Parts:**
- **WorkflowBuilder**: Di main engine wey dey design workflows
- **AIAgent**: Each specialized agent get im special skill
- **Azure OpenAI Client**: Azure OpenAI Responses API connection
- **Execution Context**: Manage state and data flow between workflow steps

## 🎨 **Enterprise Workflow Design Patterns**

### 📝 **Content Production Workflow**
```
User Request → Content Generation → Quality Review → Final Output
```

### 🔍 **Document Processing Pipeline**
```
Document Input → Analysis → Extraction → Validation → Structured Output
```

### 💼 **Business Intelligence Workflow**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

### 🤝 **Customer Service Automation**
```
Customer Inquiry → Classification → Processing → Response Generation → Follow-up
```

## 🏢 **Enterprise Benefits**

### 🎯 **Reliability & Scalability**
- **Deterministic Execution**: Workflow result fit dey same, repeatable
- **Error Recovery**: Handle failure well well for any workflow step
- **Performance Monitoring**: Check execution metrics plus chance to optimize
- **Resource Management**: Use AI model resources well without waste

### 🔒 **Security & Compliance**
- **Secure Authentication**: Microsoft Entra ID login via `az login` (AzureCliCredential)
- **Audit Trails**: Full record of workflow execution and decision points
- **Access Control**: Detailed permissions for workflow running and watching
- **Data Privacy**: Keep sensitive info secure throughout workflow

### 📊 **Observability & Management**
- **Visual Workflow Design**: Clear picture of process flow and how wetin dem depend on each other
- **Execution Monitoring**: Real-time tracking of workflow progress and how e perform
- **Error Reporting**: Detailed error check and debugging tools
- **Performance Analytics**: Metrics for optimization and plan capacity

Make we build your first enterprise-ready AI workflow! 🚀


In [ ]:
#r "nuget: Microsoft.Extensions.AI, 10.*"

In [ ]:
#r "nuget: Azure.AI.OpenAI, 2.1.0"

In [ ]:
#r "nuget: Azure.Identity, 1.15.0"
#r "nuget: System.Linq.Async, 6.0.3"
#r "nuget: OpenTelemetry.Api, 1.0.0"

In [ ]:

#r "nuget: Microsoft.Agents.AI.Workflows, 1.*"

In [ ]:

#r "nuget: Microsoft.Agents.AI.OpenAI, 1.*-*"

In [ ]:
#r "nuget: DotNetEnv, 3.1.1"

In [ ]:
// #r "nuget: Microsoft.Extensions.AI.OpenAI, 1.*-*"

In [ ]:
using System;
using System.ComponentModel;
using System.Text;
using Azure.AI.OpenAI;
using Azure.Identity;
using Microsoft.Extensions.AI;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;

In [ ]:
 using DotNetEnv;

In [ ]:
Env.Load("../../../.env");

In [ ]:
// Azure OpenAI with the Responses API (stable v1 endpoint). Sign in with `az login`.
var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT") ?? throw new InvalidOperationException("AZURE_OPENAI_ENDPOINT is not set.");
var deployment = Environment.GetEnvironmentVariable("AZURE_OPENAI_DEPLOYMENT") ?? "gpt-5-mini";

In [ ]:
// The Azure OpenAI client is created directly from the endpoint and Azure CLI credential — no custom client options are required.

In [ ]:
var azureClient = new AzureOpenAIClient(new Uri(azureEndpoint), new AzureCliCredential());

In [ ]:
const string ReviewerAgentName = "Concierge";
const string ReviewerAgentInstructions = @"
    You are a hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. ";

In [ ]:
const string FrontDeskAgentName = "FrontDesk";
const string FrontDeskAgentInstructions = @"""
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """;

In [ ]:
AIAgent reviewerAgent = azureClient.GetChatClient(deployment).AsIChatClient().AsAIAgent(
    name:ReviewerAgentName,instructions:ReviewerAgentInstructions);
AIAgent frontDeskAgent  = azureClient.GetChatClient(deployment).AsIChatClient().AsAIAgent(
    name:FrontDeskAgentName,instructions:FrontDeskAgentInstructions);

In [ ]:
var workflow = new WorkflowBuilder(frontDeskAgent)
            .AddEdge(frontDeskAgent, reviewerAgent)
            .Build();

In [ ]:
ChatMessage userMessage = new ChatMessage(ChatRole.User, [
	new TextContent("I would like to go to Paris.") 
]);

In [ ]:
StreamingRun run = await InProcessExecution.RunStreamingAsync(workflow, userMessage);

In [ ]:
await run.TrySendMessageAsync(new TurnToken(emitEvents: true));
string id="";
var messageData = new StringBuilder();
await foreach (WorkflowEvent evt in run.WatchStreamAsync().ConfigureAwait(false))
{
    if (evt is AgentResponseUpdateEvent executorComplete)
    {
        if(id=="")
        {
            id=executorComplete.ExecutorId;
        }
        if(id==executorComplete.ExecutorId)
        {
            if (executorComplete.Data is not null)
            {
                messageData.Append(executorComplete.Data.ToString());
            }
        }
        else
        {
            id=executorComplete.ExecutorId;
        }
        // Console.WriteLine($"{executorComplete.ExecutorId}: {executorComplete.Data}");
    }
}

Console.WriteLine(messageData.ToString());

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
